# 💳 *Financial Transaction Fraud Detection & Pattern Analysis*

#### A Machine Learning Approach Using Supervised and Unsupervised Learning

---

**Course:** CSD 302 — Machine Learning  
**Project Type:** Individual Course Project  
**Domain:** Financial Fraud Detection  
**Dataset:** BankSim — Synthetic Financial Transaction Dataset

## 🎯 Problem Statement
Financial fraud is a critical challenge in digital payment systems, where fraudulent transactions are typically much less frequent than legitimate ones and may exhibit complex transaction patterns.

This project aims to develop an **end-to-end machine learning framework** that:

- 🔍 **Detects fraudulent financial transactions** using supervised learning while addressing class imbalance and preventing data leakage.
- 📊 **Compares multiple classification models** using appropriate metrics such as *Precision, Recall, F1-score, ROC-AUC,* and *Precision-Recall AUC*.
- ⚙️ **Optimizes model performance** through stratified cross-validation, hyperparameter tuning, and decision-threshold analysis.
- 🧩 **Discovers natural transaction patterns** using unsupervised learning without using fraud labels during clustering.
- 📈 **Analyzes the discovered clusters** to determine whether certain transaction patterns are associated with different concentrations of fraudulent activity.
- ⚖️ **Examines practical and ethical considerations**, including class imbalance, false positives, false negatives, privacy, bias, and limitations of the dataset.

> **Overall Goal:** Build and evaluate a reliable machine learning framework for *fraud detection and transaction pattern analysis* while maintaining an honest and interpretable evaluation of model performance.

## 📊 Dataset Description

### BankSim Dataset

The project uses the **BankSim** dataset, a synthetic financial transaction dataset generated using a payment simulator designed to reproduce characteristics of real-world banking transactions.

The dataset contains approximately **594,643 transactions**, including both legitimate and fraudulent transactions.

### Main Features

| Feature | Description | Type |
|---|---|---|
| `step` | Time step associated with the transaction | Numerical |
| `customer` | Customer identifier | Identifier |
| `age` | Customer age group | Categorical |
| `gender` | Customer gender | Categorical |
| `zipcodeOri` | Origin ZIP code | Categorical/Identifier |
| `merchant` | Merchant identifier | Identifier |
| `zipMerchant` | Merchant ZIP code | Categorical/Identifier |
| `category` | Transaction category | Categorical |
| `amount` | Transaction amount | Numerical |
| `fraud` | Fraud indicator: 0 = legitimate, 1 = fraudulent | Target |

### Important Dataset Characteristics

- The target variable is `fraud`.
- Fraudulent transactions form a minority class.
- The dataset contains both numerical and categorical variables.
- Several columns represent customers, merchants, or locations and therefore require careful treatment during preprocessing.
- The dataset is synthetic, which should be considered when interpreting the results and discussing limitations.

## ⚠️ Important Evaluation Principle

Fraud detection is an **imbalanced classification problem** because legitimate transactions substantially outnumber fraudulent transactions.

Therefore, **accuracy alone is not an appropriate measure of model quality**.

For example, a model could classify almost every transaction as legitimate and still achieve a high accuracy while failing to detect fraudulent transactions.

Therefore, this project will primarily focus on:

- **Precision** — How many transactions predicted as fraud were actually fraudulent?
- **Recall** — How many actual fraudulent transactions were successfully detected?
- **F1-score** — Balance between precision and recall.
- **ROC-AUC** — Ability of the model to distinguish between the two classes across thresholds.
- **PR-AUC / Average Precision** — Particularly informative when the positive class is rare.

The final model will also be evaluated using a **confusion matrix** and decision-threshold analysis.

## 🛠️ 1. Importing Required Libraries

We begin by importing the Python libraries required for data loading, manipulation, visualization, and machine learning.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 📥 2. Loading the Dataset

The raw BankSim transaction dataset is stored as a CSV file in the project directory.

We load the dataset into a Pandas DataFrame so that it can be inspected, cleaned, visualized, and subsequently used for machine learning.

> **Important:** The raw dataset will be treated as the original source data. We will avoid modifying the raw CSV directly and will perform cleaning and transformations within the notebook.

In [3]:
data_path = Path("bs140513_032310.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 🔎 3. Initial Dataset Inspection

Before performing any cleaning or preprocessing, we first inspect the basic structure of the dataset.

In [5]:
print(f"Number of rows    : {df.shape[0]:,}")
print(f"Number of columns : {df.shape[1]}")

Number of rows    : 594,643
Number of columns : 10


In [6]:
df.head()

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594643 entries, 0 to 594642
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   step         594643 non-null  int64  
 1   customer     594643 non-null  object 
 2   age          594643 non-null  object 
 3   gender       594643 non-null  object 
 4   zipcodeOri   594643 non-null  object 
 5   merchant     594643 non-null  object 
 6   zipMerchant  594643 non-null  object 
 7   category     594643 non-null  object 
 8   amount       594643 non-null  float64
 9   fraud        594643 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 45.4+ MB


In [8]:
df.describe(include="all")

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
count,594643.000000,594643,594643,594643,594643,594643,594643,594643,594643.000000,594643.000000
unique,NaN,4112,8,4,1,50,1,15,NaN,NaN
top,NaN,'C1978250683','2','F','28007','M1823072687','28007','es_transportation',NaN,NaN
freq,NaN,265,187310,324565,594643,299693,594643,505119,NaN,NaN
mean,94.986827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.890135,0.012108
std,51.053632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,111.402831,0.109369
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,52.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.740000,0.000000
50%,97.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.900000,0.000000
75%,139.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.540000,0.000000


### 📝 Key Takeaway

The BankSim dataset contains **594,643 transactions and 10 variables**.

The initial inspection shows that:

- There are **no missing values** in any column.
- The dataset contains numerical, categorical, and identifier-style variables.
- `fraud` is a binary target variable.
- Only approximately **1.21% of transactions are fraudulent**, confirming that this is a highly imbalanced classification problem.
- `amount` shows substantial variation, while variables such as `customer`, `merchant`, and location codes represent entities rather than ordinary numerical measurements.


## 🧹 4. Data Cleaning & Quality Checks

### Checking Missing Values

In [11]:
missing_values = df.isnull().sum()
print(f"Total missing values: {missing_values.sum():,}")

Total missing values: 0


### Checking duplicate records

In [12]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count:,}")

Number of duplicate rows: 0


### Target Validation

In [14]:
print("Unique values in fraud column:")
print(sorted(df["fraud"].unique()))
print("\nFraud value counts:")
print(df["fraud"].value_counts())

Unique values in fraud column:
[np.int64(0), np.int64(1)]

Fraud value counts:
fraud
0    587443
1      7200
Name: count, dtype: int64


### Categorical Inspection

In [15]:
categorical_columns = [
    "age",
    "gender",
    "category"
]

for col in categorical_columns:
    print(f"\n{'=' * 50}")
    print(f"{col}")
    print(f"{'=' * 50}")
    print(df[col].value_counts())

print("Step range:")
print(f"Minimum: {df['step'].min()}")
print(f"Maximum: {df['step'].max()}")

print("\nAmount range:")
print(f"Minimum: {df['amount'].min()}")
print(f"Maximum: {df['amount'].max()}")
print("\nNumber of negative transaction amounts:", (df["amount"] < 0).sum())


age
age
'2'    187310
'3'    147131
'4'    109025
'5'     62642
'1'     58131
'6'     26774
'0'      2452
'U'      1178
Name: count, dtype: int64

gender
gender
'F'    324565
'M'    268385
'E'      1178
'U'       515
Name: count, dtype: int64

category
category
'es_transportation'        505119
'es_food'                   26254
'es_health'                 16133
'es_wellnessandbeauty'      15086
'es_fashion'                 6454
'es_barsandrestaurants'      6373
'es_hyper'                   6098
'es_sportsandtoys'           4002
'es_tech'                    2370
'es_home'                    1986
'es_hotelservices'           1744
'es_otherservices'            912
'es_contents'                 885
'es_travel'                   728
'es_leisure'                  499
Name: count, dtype: int64
Step range:
Minimum: 0
Maximum: 179

Amount range:
Minimum: 0.0
Maximum: 8329.96

Number of negative transaction amounts: 0


### 📝 Data Cleaning — Key Takeaway

The data-quality checks show that the dataset is structurally clean:

- No missing values were detected.
- No duplicate rows were found.
- The `fraud` column contains only valid binary values (`0` and `1`).
- No negative transaction amounts were detected.
- The `step` and `amount` variables fall within valid observed ranges.

However, several categorical variables contain special categories such as `U` and `E`. These values are not missing values and should therefore not be removed automatically. Their treatment will be considered during preprocessing.

The target distribution confirms a strong class imbalance, with only **7,200 fraudulent transactions compared with 587,443 legitimate transactions**. This imbalance will be explicitly addressed during the machine learning stage.

> **Conclusion:** No major row-level cleaning is required at this stage. We can now proceed to **Exploratory Data Analysis (EDA)**.